# BKQuiz: CPT Qwen3 1.7B — 27 học phần
Chọn **Python 3 / GPU**, ưu tiên L4. Dữ liệu và model khác nhau: Qwen3 1.7B là model train; VLM preprocessing đã chạy trước đó.

Đặt `bkquiz_cpt_27courses_20261023.zip` trong **MyDrive/BKQuiz_CPT**. Chạy các cell theo thứ tự.
Đợt này giữ các quyết định VLM đã lưu; dữ liệu chưa từng duyệt được chấp nhận theo chỉ thị chủ dữ liệu. Xem `AUDIT_REPORT.md` trong ZIP để biết giới hạn chất lượng.
Đây là **QLoRA CPT dự đoán token tiếp theo**, 1 epoch, không áp dụng chat template, không phải SFT hỏi–đáp hoặc full-parameter training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/BKQuiz_CPT')
ZIP_PATH = DRIVE_ROOT / 'bkquiz_cpt_27courses_20261023.zip'
assert ZIP_PATH.is_file(), f'Chưa tìm thấy {ZIP_PATH}'


In [ ]:
import hashlib, json, zipfile, shutil, os, subprocess, sys
EXPECTED_ZIP_SHA256 = '6da7e92a75a4d9291ce23697fb9dfa1a309cf99b289f230bee50fa833919ac62'
LOCAL_ZIP = Path('/content/bkquiz_cpt_27courses_20261023.zip')
shutil.copyfile(ZIP_PATH, LOCAL_ZIP)
def file_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(1024*1024), b''): digest.update(block)
    return digest.hexdigest()
assert file_sha(LOCAL_ZIP) == EXPECTED_ZIP_SHA256, 'ZIP/notebook không cùng bản.'
WORK = Path('/content/bkquiz_cpt_27courses')
with zipfile.ZipFile(LOCAL_ZIP) as archive:
    for entry in archive.infolist():
        assert (Path('/content') / entry.filename).resolve().is_relative_to(WORK.resolve()), 'Đường dẫn ZIP không hợp lệ'
    archive.extractall('/content')
os.chdir(WORK)
bundle = json.loads((WORK / 'bundle_manifest.json').read_text())
for name, digest in bundle['files_sha256'].items():
    assert file_sha(WORK / name) == digest, f'File bị thay đổi: {name}'
data = json.loads((WORK / 'data/manifest.json').read_text())
assert data['data_ready_for_owner_authorized_training'] and data['protected_splits_excluded']
print('Mẫu:', data['samples'], '| Học phần:', len(data['courses']), '| Ký tự:', data['characters'])
print('Chấp nhận:', data['acceptance'])


## Môi trường và GPU
Tạo venv riêng, dùng PyTorch CUDA có sẵn của Colab và khóa các thư viện còn lại. Không cài Ollama.
Chạy kiểm thử dữ liệu trước khi nạp model. Nếu GPU hoặc môi trường lỗi, cell dừng trước train.

In [ ]:
VENV = Path('/content/bkquiz_cpt_venv')
if not (VENV / 'bin/python').exists():
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', str(VENV)], check=True)
PYTHON = str(VENV / 'bin/python')
subprocess.run([PYTHON, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
subprocess.run([PYTHON, '-m', 'unittest', 'discover', '-s', '.', '-p', 'test_data.py', '-v'], check=True)
subprocess.run(['nvidia-smi'], check=True)
subprocess.run([PYTHON, '-c', "import torch,transformers,peft,bitsandbytes; from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments; from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training; from packaging.version import Version; assert torch.cuda.is_available(), 'Chon GPU runtime'; assert Version(torch.__version__.split('+')[0])>=Version('2.6'); print('GPU:',torch.cuda.get_device_name(0),'torch:',torch.__version__,'transformers:',transformers.__version__)"], check=True)
def launch(script, *arguments):
    subprocess.run([PYTHON, script, *map(str, arguments)], check=True)


## Tokenize và packing
Tokenizer đúng revision Qwen3; EOS phân cách mẫu. Packing giữ token cuối, mask padding bằng vị trí, labels chưa shift.
Chỉ xuất các split train đã có; không chia ngẫu nhiên trang thành test mới. Không có phép đo tổng quát hóa độc lập trong notebook này.

In [ ]:
PREPARED = Path('/content/bkquiz_cpt_prepared')
launch('prepare.py', '--out', PREPARED)
stats = json.loads((PREPARED / 'manifest.json').read_text())
print(json.dumps(stats, indent=2, ensure_ascii=False))
cfg = json.loads((WORK / 'config.json').read_text())
import math
print('Optimizer steps/epoch:', math.ceil(stats['packed_sequences'] / (cfg['batch_size'] * cfg['gradient_accumulation_steps'])))


## Tên đợt train và tiếp tục checkpoint
Giữ nguyên RUN_NAME khi tiếp tục. Đổi tên khi đổi dữ liệu, cấu hình hoặc làm thí nghiệm mới.
Checkpoint và adapter nằm trên Drive; 25 optimizer steps lưu một checkpoint, giữ hai bản gần nhất.

In [ ]:
RUN_NAME = 'cpt_27courses_20261023_001'
RUN_DIR = DRIVE_ROOT / 'runs' / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
identity = {'zip_sha256': EXPECTED_ZIP_SHA256, 'dataset_manifest_sha256': file_sha(WORK / 'data/manifest.json')}
identity_path = RUN_DIR / 'bundle_identity.json'
if identity_path.exists():
    assert json.loads(identity_path.read_text()) == identity, 'RUN_NAME thuộc dữ liệu/code khác; chọn tên mới.'
else:
    identity_path.write_text(json.dumps(identity, indent=2))
RESUME_SMOKE = False
RESUME_TRAIN = False  # True khi tiếp tục cùng đợt bị ngắt.


## Smoke test: 2 optimizer steps
Kiểm tra loss hữu hạn, gradient, LoRA đổi trọng số và checkpoint. Lượt train chính khởi tạo lại từ model gốc; không dùng adapter smoke.

In [ ]:
smoke_args = ['--prepared', PREPARED, '--output', RUN_DIR / 'smoke', '--mode', 'smoke']
if RESUME_SMOKE: smoke_args += ['--resume', 'latest']
if not (RUN_DIR / 'smoke/completed.json').exists(): launch('train.py', *smoke_args)
smoke = json.loads((RUN_DIR / 'smoke/completed.json').read_text())
assert math.isfinite(smoke['metrics']['train_loss'])
assert smoke['lora_changed_vs_initial'] is not False
print(smoke)


## CPT chính thức: 1 epoch
QLoRA NF4, sequence 1024, batch 1, accumulation 8, LR 2e-5, cosine, warmup 3%, r=8.
Chẩn đoán loss trước/sau dùng cùng tối đa 64 sequence train: chỉ đo mức khớp tập train, không phải validation/test.

In [ ]:
assert (RUN_DIR / 'smoke/completed.json').is_file(), 'Smoke test chưa hoàn thành.'
train_args = ['--prepared', PREPARED, '--output', RUN_DIR / 'train', '--mode', 'train']
if RESUME_TRAIN: train_args += ['--resume', 'latest']
if not (RUN_DIR / 'train/completed.json').exists(): launch('train.py', *train_args)
result = json.loads((RUN_DIR / 'train/completed.json').read_text())
print(json.dumps(result, indent=2))
ADAPTER = RUN_DIR / 'train/adapter'
assert (ADAPTER / 'adapter_model.safetensors').is_file()
print('Adapter đã lưu:', ADAPTER)


## Tùy chọn: quiz trước/sau và model HF đã merge
Probe chỉ là cùng sáu tình huống IT3170 có sẵn, không phải benchmark độc lập. Mặc định tắt để tập trung train.
Merge cần thêm RAM/dung lượng Drive; adapter có thể dùng trực tiếp cùng base, không bắt buộc merge.

In [ ]:
RUN_PROBES = False
EXPORT_MERGED = False
if RUN_PROBES:
    if not (RUN_DIR / 'baseline_quiz.json').exists(): launch('probe.py', '--out', RUN_DIR / 'baseline_quiz.json')
    if not (RUN_DIR / 'after_cpt_quiz.json').exists(): launch('probe.py', '--adapter', ADAPTER, '--out', RUN_DIR / 'after_cpt_quiz.json')
if EXPORT_MERGED:
    launch('merge_adapter.py', '--adapter', ADAPTER, '--out', RUN_DIR / 'merged_hf_fp16')


Khi Colab bị ngắt: kết nối lại GPU, chạy lại mount/extract/install/prepare, giữ RUN_NAME và đặt RESUME_TRAIN=True.
Trainer chỉ nhận checkpoint đầy đủ, cùng data/config/code/thư viện. Adapter smoke không phải kết quả cuối.
Chạy xong thì ngắt runtime. Notebook không tự đẩy model lên Hugging Face và không sửa dữ liệu preprocessing hay lịch sử duyệt.